# 08 Feature Visualization (PCA / t-SNE)

Project trained bottleneck features to 2D to visualize class separability.

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU Auto-Detection ───────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"Detected number of GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (No GPU — running on CPU)")


In [ ]:
import matplotlib.pyplot as plt
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from src.experiments import (
    load_cfg, build_trainer_from_checkpoint, build_loaders, collect_features
)

cfg = load_cfg("configs/cifar10.yaml")
checkpoint_path = "outputs/checkpoints/cifar10_full_sdd_last.pt"
trainer = build_trainer_from_checkpoint(cfg, checkpoint_path)
_, test_loader = build_loaders(cfg)

feats, labels = collect_features(trainer, test_loader)
feats_np  = feats.numpy()
labels_np = labels.numpy()
print(f"features shape: {feats_np.shape}")


In [ ]:
CIFAR10_CLASSES = ["airplane","automobile","bird","cat","deer",
                   "dog","frog","horse","ship","truck"]
import matplotlib.cm as cm

pca = PCA(n_components=2).fit_transform(feats_np)
plt.figure(figsize=(7, 6))
colors = cm.get_cmap("tab10", 10)
for c in range(10):
    mask = labels_np == c
    plt.scatter(pca[mask, 0], pca[mask, 1], s=8, alpha=0.6,
                color=colors(c), label=CIFAR10_CLASSES[c])
plt.legend(markerscale=2, fontsize=8, loc="best")
plt.title("PCA of bottleneck features (Full SDD)")
plt.tight_layout()
plt.savefig("outputs/figures/pca_features.png", dpi=150)
plt.show()


In [ ]:
idx = torch.randperm(len(feats_np))[:2000]
tsne = TSNE(n_components=2, init="pca", learning_rate="auto",
            perplexity=30, random_state=42).fit_transform(feats_np[idx])
l2   = labels_np[idx]

plt.figure(figsize=(7, 6))
for c in range(10):
    mask = l2 == c
    plt.scatter(tsne[mask, 0], tsne[mask, 1], s=8, alpha=0.6,
                color=cm.get_cmap("tab10", 10)(c), label=CIFAR10_CLASSES[c])
plt.legend(markerscale=2, fontsize=8, loc="best")
plt.title("t-SNE of bottleneck features (Full SDD)")
plt.tight_layout()
plt.savefig("outputs/figures/tsne_features.png", dpi=150)
plt.show()
